Unique Genre

In [ ]:
print("\n3. Explore Data & 4. Preprocessing...")
df.replace('N/A', np.nan, inplace=True)
df.dropna(subset=['artist_genres'], inplace=True) 

all_genres = set()
for genre_string in df['artist_genres'].dropna():
    genres = [g.strip() for g in genre_string.split(',')]
    all_genres.update(genres)
unique_genres = sorted(list(all_genres))

def assign_mood(row):
    pop = row['track_popularity']
    dur = row['track_duration_min']
    if pop > 50 and dur > 3.0: return 'Energetic'
    elif pop > 50 and dur <= 3.0: return 'Happy'
    elif pop <= 50 and dur > 3.0: return 'Chill'
    else: return 'Sad'

df['mood'] = df.apply(assign_mood, axis=1)
unique_moods = sorted(df['mood'].unique().tolist())
print("Preprocessing complete!")

In [ ]:
print("\n--- 🎵 Music Generator UI 🎵 ---")

mood_dropdown = widgets.Dropdown(options=['Any'] + unique_moods, value='Any', description='Mood:')
genre_dropdown = widgets.Dropdown(options=['Any'] + unique_genres, value='Any', description='Genre:')

generate_button = widgets.Button(description="Generate Music", button_style='success', icon='music')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        selected_genre = genre_dropdown.value
        selected_mood = mood_dropdown.value
        
        filtered_df = df.copy()
        if selected_genre != 'Any':
            filtered_df = filtered_df[filtered_df['artist_genres'].str.contains(selected_genre, case=False, na=False)]
        if selected_mood != 'Any':
            filtered_df = filtered_df[filtered_df['mood'] == selected_mood]
            
        if filtered_df.empty:
            print(f"❌ No tracks found for Genre: {selected_genre} and Mood: {selected_mood}. Try a different combination!")
        else:
            print(f"✅ Generated tracks for Genre: '{selected_genre}' & Mood: '{selected_mood}'!\n")
            sample_size = min(3, len(filtered_df))
            recommendations = filtered_df.sample(sample_size)
            
            # MEMBUAT PEMUTAR AUDIO UNTUK SETIAP LAGU
            for idx, row in recommendations.iterrows():
                track_id = row['track_id']
                
                # Membuat HTML Iframe untuk Spotify Player
                spotify_player = f"""
                <iframe src="https://open.spotify.com/embed/track/{track_id}" 
                        width="300" height="80" frameborder="0" 
                        allowtransparency="true" allow="encrypted-media">
                </iframe>
                <br>
                """
                # Menampilkan pemutar musik
                display(HTML(spotify_player))

generate_button.on_click(on_button_clicked)
display(mood_dropdown, genre_dropdown, generate_button, output)